# Demo — Cost Controls and Runaway Execution Prevention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-cost-controls/demo-cost-controls.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 6: Best Practices and Wrap-Up

Demonstrates infrastructure-level controls to prevent runaway costs:
idle timeouts, max iterations, circuit breakers, and proactive session
termination. Shows why deterministic limits beat prompt-level instructions.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def show_cost_dimensions():
    """AgentCore costs span multiple billing dimensions."""
    print_section("Emergent Cost Dimensions")

    costs = {
        "Runtime compute": "Per-second billing for active session microVMs",
        "Gateway API calls": "Per-request pricing for tool invocations",
        "Memory operations": "Read/write operations on Memory resources",
        "Identity broker": "Token issuance and credential exchange",
        "LLM inference tokens": "Input + output tokens per model invocation",
    }
    print(f"  {json.dumps(costs, indent=2)}")
    print()
    print("  Cost amplification: one user request can trigger:")
    print("  - Multiple LLM calls (reasoning → tool choice → response)")
    print("  - Multiple tool invocations (each with Gateway cost)")
    print("  - Multiple memory reads/writes")
    print("  → Rate limiting must account for this amplification factor")

def show_idle_timeout():
    """Idle timeout and max lifetime prevent idle instance billing."""
    print_section("Lifecycle Controls — Idle Timeout & Max Lifetime")

    config = {
        "idleRuntimeSessionTimeout": 300,  # 5 minutes
        "maxLifetime": 28800,  # 8 hours
        "pattern": "Call StopRuntimeSession API when task completes",
    }

    print("  These are set in the Runtime lifecycle configuration:")
    print(f"  {json.dumps(config, indent=2)}")
    print()
    print("  Without these controls:")
    print("  - Idle sessions continue billing indefinitely")
    print("  - A forgotten session could run for days, costing hundreds of dollars")
    print()
    print("  Best practice: Set idle-timeout to the minimum acceptable for your use case.")
    print("  Call StopRuntimeSession proactively — don't wait for the timeout.")

def show_runaway_execution():
    """Demonstrate hallucination loops and deterministic limits."""
    print_section("Runaway Execution — Hallucination Loops")

    print("  A hallucination loop occurs when an agent:")
    print("  1. Calls a tool → tool returns error")
    print("  2. Agent retries with slightly different parameters → same error")
    print("  3. Agent retries again → same error")
    print("  4. Repeats indefinitely, consuming tokens and API calls")
    print()
    print("  Deterministic controls to prevent this:")
    controls = {
        "max_iterations": "Hard cap on reasoning steps per request (e.g., 50)",
        "harness_timeout": "Wall-clock timeout per request (e.g., 600s)",
        "circuit_breaker": "Halt if N consecutive tool calls fail",
        "spend_budget": "Max cost per request (tokens × model price)",
        "idempotency_keys": "Prevent duplicate side effects on retry",
    }
    print(f"  {json.dumps(controls, indent=2)}")
    print()
    print("  CRITICAL: Do NOT rely on prompt instructions to prevent loops.")
    print('  "Stop trying if it fails" in a system prompt is NOT a control.')
    print("  max_iterations in the infrastructure config IS a control.")

def show_prompt_caching():
    """Prompt caching reduces token costs for repeated prefixes."""
    print_section("Token Optimization — Prompt Caching")

    print("  Prompt caching stores frequently-used prompt prefixes:")
    print("  - System prompts (reused across all requests)")
    print("  - Tool definitions (reused across all tool calls)")
    print("  - Few-shot examples (reused across similar requests)")
    print()
    print("  Savings example:")
    print("  - Without caching: 1000 input tokens × $0.003/1K = $0.003 per request")
    print("  - With caching: 200 input tokens × $0.003/1K + 800 cached × $0.000375/1K")
    print("  - Savings: ~70% on input token costs for repeated prefixes")
    print()
    print("  NOTE: Prompt caching is an instructor talking point.")
    print("  Students do not configure caching in the 4-hour lab.")

def main():
    print("Cost Controls and Runaway Prevention — Instructor Demo\n")
    show_cost_dimensions()
    show_idle_timeout()
    show_runaway_execution()
    show_prompt_caching()
    print_section("Key Takeaways")
    print("  1. Agent costs span 5 dimensions — monitor all of them")
    print("  2. idle-timeout and max-lifetime prevent idle billing")
    print("  3. max-iterations and harness-timeout are infrastructure controls, not prompts")
    print("  4. Circuit breakers halt cascading failures at the Gateway layer")
    print("  5. Prompt caching can reduce token costs by up to 70%")

if __name__ == "__main__":
    main()
